In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display


# PROJECT PATHS


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

EVALUATION_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "environmental_anomalies"
    / "ifs_baseline_evaluation_2024_2025"
)

REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "environmental_anomalies"
    / "ifs_baseline_evaluation_2024_2025"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)


# DISPLAY SETTINGS


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

sns.set_theme(style="whitegrid")

print("Project directory:", PROJECT_DIR)
print("Evaluation directory:", EVALUATION_DIR)
print("Report directory:", REPORT_DIR)

print("\nNotebook environment configured successfully.")

Project directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave
Evaluation directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\environmental_anomalies\ifs_baseline_evaluation_2024_2025
Report directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave\reports\environmental_anomalies\ifs_baseline_evaluation_2024_2025

Notebook environment configured successfully.


In [2]:

# CELL 2: LOAD EVALUATION DATASETS

# Define file paths
TEST_FILE = EVALUATION_DIR / "ifs_anomaly_test_2024_2025.csv"
METRICS_FILE = EVALUATION_DIR / "ifs_anomaly_evaluation_metrics.csv"
DISTRICT_FILE = EVALUATION_DIR / "ifs_anomaly_district_metrics.csv"
MONTHLY_FILE = EVALUATION_DIR / "ifs_anomaly_monthly_metrics.csv"

# Check whether all files exist
files = {
    "Test classifications": TEST_FILE,
    "Overall metrics": METRICS_FILE,
    "District metrics": DISTRICT_FILE,
    "Monthly metrics": MONTHLY_FILE
}

print("=" * 65)
print("EVALUATION FILE VALIDATION")
print("=" * 65)

for name, path in files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

missing_files = [
    name for name, path in files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing evaluation files: {missing_files}"
    )

# Load datasets
test_df = pd.read_csv(TEST_FILE, parse_dates=["date"])
metrics_df = pd.read_csv(METRICS_FILE)
district_metrics_df = pd.read_csv(DISTRICT_FILE)
monthly_metrics_df = pd.read_csv(MONTHLY_FILE)

# Display dataset dimensions
print("\n" + "=" * 65)
print("DATASET DIMENSIONS")
print("=" * 65)

datasets = {
    "Test classifications": test_df,
    "Overall metrics": metrics_df,
    "District metrics": district_metrics_df,
    "Monthly metrics": monthly_metrics_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

# Display column names
print("\n" + "=" * 65)
print("DATASET COLUMNS")
print("=" * 65)

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())

print("\nAll four evaluation datasets loaded successfully.")

EVALUATION FILE VALIDATION
Test classifications: FOUND
Overall metrics: FOUND
District metrics: FOUND
Monthly metrics: FOUND

DATASET DIMENSIONS
Test classifications: (18275, 9)
Overall metrics: (3, 12)
District metrics: (75, 13)
Monthly metrics: (36, 13)

DATASET COLUMNS

Test classifications
['district', 'date', 'rainfall_era5_anomaly', 'temperature_era5_anomaly', 'humidity_era5_anomaly', 'rainfall_ifs_anomaly', 'temperature_ifs_anomaly', 'humidity_ifs_anomaly', 'month']

Overall metrics
['variable', 'records', 'reference_anomalies', 'comparison_anomalies', 'true_positive', 'true_negative', 'false_positive', 'false_negative', 'overall_agreement', 'precision', 'recall', 'f1_score']

District metrics
['records', 'reference_anomalies', 'comparison_anomalies', 'true_positive', 'true_negative', 'false_positive', 'false_negative', 'overall_agreement', 'precision', 'recall', 'f1_score', 'district', 'variable']

Monthly metrics
['records', 'reference_anomalies', 'comparison_anomalies', 'true

In [3]:
# ==================================================
# CELL 3: EVALUATION DATA QUALITY CHECKS
# ==================================================

print("=" * 65)
print("EVALUATION DATA QUALITY CHECKS")
print("=" * 65)

# --------------------------------------------------
# 1. Missing values
# --------------------------------------------------

print("\n1. MISSING VALUE ANALYSIS")

for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    print(f"{name}: {missing} missing values")

# --------------------------------------------------
# 2. Duplicate records
# --------------------------------------------------

print("\n2. DUPLICATE RECORD ANALYSIS")

print(
    "Test district-date duplicates:",
    test_df.duplicated(["district", "date"]).sum()
)

print(
    "Duplicate overall metric variables:",
    metrics_df.duplicated(["variable"]).sum()
)

print(
    "Duplicate district-variable combinations:",
    district_metrics_df.duplicated(["district", "variable"]).sum()
)

print(
    "Duplicate month-variable combinations:",
    monthly_metrics_df.duplicated(["month", "variable"]).sum()
)

# --------------------------------------------------
# 3. Metric range validation
# --------------------------------------------------

metric_columns = [
    "overall_agreement",
    "precision",
    "recall",
    "f1_score"
]

print("\n3. METRIC RANGE VALIDATION")

for name, df in datasets.items():
    if all(col in df.columns for col in metric_columns):
        invalid = (
            (df[metric_columns] < 0) |
            (df[metric_columns] > 1)
        ).sum().sum()

        print(f"{name}: {invalid} values outside [0, 1]")

# --------------------------------------------------
# 4. Confusion matrix count validation
# --------------------------------------------------

print("\n4. CONFUSION MATRIX VALIDATION")

for name, df in {
    "Overall": metrics_df,
    "District": district_metrics_df,
    "Monthly": monthly_metrics_df
}.items():

    total = (
        df["true_positive"]
        + df["true_negative"]
        + df["false_positive"]
        + df["false_negative"]
    )

    invalid = (total != df["records"]).sum()

    print(f"{name}: {invalid} inconsistent records")

# --------------------------------------------------
# 5. Classification balance
# --------------------------------------------------

print("\n5. TEST CLASSIFICATION BALANCE")

flag_columns = [
    "rainfall_era5_anomaly",
    "temperature_era5_anomaly",
    "humidity_era5_anomaly",
    "rainfall_ifs_anomaly",
    "temperature_ifs_anomaly",
    "humidity_ifs_anomaly"
]

display(
    test_df[flag_columns]
    .sum()
    .rename("Anomaly_Count")
    .to_frame()
)

print("\nData quality checks completed.")

EVALUATION DATA QUALITY CHECKS

1. MISSING VALUE ANALYSIS
Test classifications: 0 missing values
Overall metrics: 0 missing values
District metrics: 0 missing values
Monthly metrics: 0 missing values

2. DUPLICATE RECORD ANALYSIS
Test district-date duplicates: 0
Duplicate overall metric variables: 0
Duplicate district-variable combinations: 0
Duplicate month-variable combinations: 0

3. METRIC RANGE VALIDATION
Overall metrics: 0 values outside [0, 1]
District metrics: 0 values outside [0, 1]
Monthly metrics: 0 values outside [0, 1]

4. CONFUSION MATRIX VALIDATION
Overall: 0 inconsistent records
District: 0 inconsistent records
Monthly: 0 inconsistent records

5. TEST CLASSIFICATION BALANCE


,Anomaly_Count
rainfall_era5_anomaly,326
temperature_era5_anomaly,1158
humidity_era5_anomaly,502
rainfall_ifs_anomaly,569
temperature_ifs_anomaly,1578
humidity_ifs_anomaly,1166



Data quality checks completed.
